# Step 9 (rebuilt for D-079) — Reporter

D-079 froze `service_target` at the ABC default and removed it as a decision — `line_policy_brief()` (built for D-071's 2-lever output, columns `service_A/B/C`) no longer matches the search's real output shape and is retired. `line_policy_brief_three_lever()` replaces it: cover per class (the real decision), `service_achieved` per class (the real outcome, not set), `bias_correction`/`min_run_hours` at line level.

`avoidable_cost_view`, `owner_view`, `capacity_warning`, `portfolio_brief` are unchanged — none depend on Step 7's lever shape.

Prerequisite: Steps 4, 5a, 7 (D-079 rebuild) and 8 have already been run.

## Setup

In [ ]:
import subprocess, os, sys
def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0: print("STDERR:", r.stderr[-1500:])
    return r
REPO = '/content/ibp-tradeoff'
os.chdir('/content'); sh('rm -rf ibp-tradeoff')
sh('git clone https://github.com/rdelolmog-creator/ibp-tradeoff.git')
os.chdir(REPO); sys.path.insert(0, REPO)
print('cwd:', os.getcwd())

## Rebuild Step 4, upload Step 5a, Step 7 (D-079) and Step 8 outputs

In [ ]:
import pandas as pd, numpy as np, yaml
from src.ingest import DataIngestor
from src.cleaner import DataCleaner
if not os.path.isdir('data_primary/raw'):
    sh('python generate_data.py'); sh('mv data data_primary')
ing = DataIngestor(repo_root=REPO, data_root='data_primary')
clean_master, sku_master, _ = DataCleaner(ing.schema, ing.assumptions).clean(ing.load())
os.makedirs('data_primary/clean', exist_ok=True)
clean_master.to_parquet('data_primary/clean/clean_master.parquet', index=False)
sku_master.to_parquet('data_primary/clean/sku_master.parquet', index=False)
print('clean_master:', clean_master.shape, '| sku_master:', sku_master.shape)

In [ ]:
from google.colab import files
import shutil
print('Select demand_characteristics.csv AND censoring_diagnostics.csv (Step 5a):')
uploaded = files.upload()
demand_characteristics = pd.read_csv('demand_characteristics.csv').set_index('sku_id')
censoring_diagnostics  = pd.read_csv('censoring_diagnostics.csv').set_index('sku_id')
from src.portfolio_impact import get_flagged_skus
flagged = get_flagged_skus(censoring_diagnostics.reset_index())
for f in ('demand_characteristics.csv', 'censoring_diagnostics.csv'):
    shutil.copy(f, f'data_primary/clean/{f}')
print(f'flagged: {len(flagged)}')

In [ ]:
print('Select line_results_three_lever.csv (Step 7, D-079) and '
      'step08_portfolio_summary.csv, step08_lever_consistency.csv (Step 8):')
uploaded2 = files.upload()
line_results            = pd.read_csv('line_results_three_lever.csv')
portfolio_summary       = pd.read_csv('step08_portfolio_summary.csv')
lever_consistency_table = pd.read_csv('step08_lever_consistency.csv')
print('line_results (D-079):', line_results.shape)
assert 'service_A' not in line_results.columns, 'this looks like the OLD (D-071) line_results — re-export from the rebuilt Step 7 notebook'

## Build the engine

In [ ]:
from src.engine import TradeOffEngine, LeverSettings, build_line_master
assumptions = yaml.safe_load(open('config/assumptions.yaml'))
schema      = yaml.safe_load(open('config/schema.yaml'))
engine = TradeOffEngine(assumptions, schema, clean_master, sku_master,
                        demand_characteristics, flagged)
line_master = build_line_master(assumptions, schema)
print('assumption fingerprint:', engine.assumption_fingerprint)

## Avoidable cost view — the Step 6 gate scenario

Unchanged from every earlier version — D-079 does not touch this function.

In [ ]:
from src.reporter import (avoidable_cost_view, owner_view, capacity_warning,
                          line_policy_brief_three_lever, portfolio_brief,
                          delta_vs_base_view, export_artefacts)

MVD_LINE = 'L3'
cat = str(engine.sku_master.loc[engine.line_skus(MVD_LINE)[0], 'category'])
base = LeverSettings.defaults_per_class(assumptions, cat)
gate_scenario = engine.run_scenario(MVD_LINE, base)

avoidable = avoidable_cost_view(gate_scenario, assumptions, schema)
for k in ('lost_sales_eur','excess_obsolescence_eur','working_capital_cost_eur',
          'conversion_cost_avoidable_eur','conversion_cost_fixed_eur'):
    print(f'{k:<30}{avoidable[k]:>16,.0f}')
print(f'{"total_avoidable_cost_eur":<30}{avoidable["total_avoidable_cost_eur"]:>16,.0f}')
print(f'{"total_reported_cost_eur":<30}{avoidable["total_reported_cost_eur"]:>16,.0f}')

## Owner view

In [ ]:
owners = owner_view(avoidable, assumptions)
print(owners.to_string(index=False))

## Capacity warning — Step 8 reference (static, unchanged)

In [ ]:
ps_indexed = portfolio_summary.set_index('line_id')
tests = [('L2', 10.0), ('L2', 4.0), ('L3', 10.0)]
for line_id, cover in tests:
    msg = capacity_warning(line_id, cover, ps_indexed.loc[line_id])
    print(f'{line_id} @ cover={cover}: {msg or "(no warning)"}')

## Line policy brief — D-079 shape

One row per (line, class): cover is the real decision; service_achieved is
the real outcome, not set. bias_correction / min_run_hours are line-level,
repeated across each class's row.

In [ ]:
lpb = line_policy_brief_three_lever(line_results)
print(lpb.round(4).to_string(index=False))

## Delta vs base, fixed absorption stripped (D-079 Change 5)

In [ ]:
FIXED_CONV = float(assumptions['plant_economics']['fixed_absorption_eur_line_month']) * 12
deltas = pd.DataFrame([
    delta_vs_base_view(row.to_dict(), base_total_eur=row.default_total_cost_eur,
                       fixed_conversion_eur=FIXED_CONV)
    for _, row in line_results.iterrows()
])
print(deltas.round(2).to_string(index=False))

## Portfolio brief

In [ ]:
brief_text = portfolio_brief(portfolio_summary, lever_consistency_table)
print(brief_text)

## Export the artefacts

In [ ]:
paths = export_artefacts(avoidable, lpb, deltas, brief_text, out_dir='.')
print(paths)

## Tests

In [ ]:
sh('python -m pytest tests/test_reporter.py -q --no-header')
sh('python -m pytest tests/test_engine.py tests/test_policy_model.py '
  'tests/test_portfolio_sweep.py tests/test_pipeline.py -q --no-header')

## Consolidated report — the only cell to copy

In [ ]:
import hashlib, subprocess

t_rep  = subprocess.run('python -m pytest tests/test_reporter.py -q --no-header',
                        shell=True, capture_output=True, text=True)
t_eng  = subprocess.run('python -m pytest tests/test_engine.py -q --no-header',
                        shell=True, capture_output=True, text=True)
t_pol  = subprocess.run('python -m pytest tests/test_policy_model.py -q --no-header',
                        shell=True, capture_output=True, text=True)
t_port = subprocess.run('python -m pytest tests/test_portfolio_sweep.py -q --no-header',
                        shell=True, capture_output=True, text=True)
if not os.path.isdir('data_control/raw'):
    subprocess.run('cp config/assumptions.yaml config/_bk.yaml && '
                   'cp config/assumptions_lowcensoring.yaml config/assumptions.yaml && '
                   'python generate_data.py && '
                   'cp config/_bk.yaml config/assumptions.yaml && rm config/_bk.yaml && '
                   'mv data data_control', shell=True, capture_output=True, text=True)
t_pipe = subprocess.run('python -m pytest tests/test_pipeline.py -q --no-header',
                        shell=True, capture_output=True, text=True)

L=[]; w=L.append
w('='*78); w('STEP 9 (D-079) - REPORTER - CONSOLIDATED REPORT'); w('='*78)
w(f'assumption set   : {engine.assumption_fingerprint}')
w(f'reporter.py sha  : {hashlib.sha256(open("src/reporter.py","rb").read()).hexdigest()[:12]}')
w(f'pandas {pd.__version__} / numpy {np.__version__}')

w(''); w('-- 1. AVOIDABLE COST VIEW - Step 6 gate scenario '+'-'*28)
for k in ('lost_sales_eur','excess_obsolescence_eur','working_capital_cost_eur',
          'conversion_cost_avoidable_eur','conversion_cost_fixed_eur',
          'total_avoidable_cost_eur','total_reported_cost_eur'):
    w(f'{k:<32}{avoidable[k]:>16,.0f}')

w(''); w('-- 2. OWNER VIEW '+'-'*61)
w(owners.to_string(index=False))

w(''); w('-- 3. LINE POLICY BRIEF, D-079 shape '+'-'*41)
w(lpb.round(4).to_string(index=False))

w(''); w('-- 4. DELTA VS BASE, fixed absorption stripped '+'-'*30)
w(deltas.round(2).to_string(index=False))

w(''); w('-- 5. PORTFOLIO BRIEF '+'-'*56)
w(brief_text)

w(''); w('-- 6. TESTS '+'-'*66)
for label, r in (('test_reporter.py', t_rep), ('test_engine.py', t_eng),
                 ('test_policy_model.py', t_pol), ('test_portfolio_sweep.py', t_port),
                 ('test_pipeline.py', t_pipe)):
    w(f'{label:<24}: ' + (r.stdout.strip().splitlines() or ["no output"])[-1])
if any(r.returncode for r in (t_rep,t_eng,t_pol,t_port,t_pipe)):
    w(''); w('FAILURES:')
    for r in (t_rep,t_eng,t_pol,t_port,t_pipe):
        if r.returncode: w(r.stdout[-2500:])

w(''); w('-- 7. CHECKS '+'-'*65)
checks = [
 ('both avoidable and reported totals present',
  'total_avoidable_cost_eur' in avoidable and 'total_reported_cost_eur' in avoidable),
 ('reported total exceeds avoidable total', avoidable['total_reported_cost_eur'] > avoidable['total_avoidable_cost_eur']),
 ('line_policy_brief has 3 rows per line, no service decision column',
  len(lpb) == lpb.line_id.nunique() * 3 and 'service_target' not in lpb.columns),
 ('bias/min_run repeat per line across classes',
  bool(lpb.groupby('line_id')['bias_correction'].nunique().eq(1).all())),
 ('delta table present for every line', len(deltas) == line_results.line_id.nunique()),
 ('owner mapping present and used', len(owners) > 0),
 ('all test suites pass', all(r.returncode==0 for r in (t_rep,t_eng,t_pol,t_port,t_pipe))),
]
for label, ok in checks:
    w(f'  [{"PASS" if ok else "SEE NOTE"}]  {label}')
w('')
w('Magnitudes are not findings (arch section 10). Every number above is')
w('whatever the generator and the assumption set encoded.')
w('='*78)

report_text = '\n'.join(L)
open('step09_d079_report.txt','w').write(report_text)
try:
    import shutil
    d = '/content/drive/My Drive/ibp-tradeoff-outputs'
    if os.path.isdir(d):
        for f in ('step09_d079_report.txt','avoidable_cost_summary.csv',
                  'line_policy_brief.csv','class_breakdown.csv','portfolio_brief.txt'):
            if os.path.exists(f): shutil.copy(f, d)
        print('saved to', d, '\\n')
except Exception as e:
    print('Drive copy skipped:', e, '\\n')
print(report_text)